# Run simulations across options

Note: if too many simultaneous subprocesses are created when not running with `gnu parallel`, they will not survive the closure of this notebook and continue to run in the background.

## Select the simulations

In [ ]:
import subprocess
from lucifex.io import create_dir_path, dir_path_exists
from lucifex.sim.parallel import combine_options
from lucifex.utils.py_utils import FrozenDict
from crocodil.dns.theory import create_Nx_selector
from crocodil.dns.system_a import SYSTEM_A_REFERENCE

Ra_ref = SYSTEM_A_REFERENCE['Ra']
Ra_opts = tuple(i * Ra_ref for i in (1.0, 0.5, 2.0, 0.1)[-1:])
Da_ref = SYSTEM_A_REFERENCE['Da']
Da_opts = tuple(i * Da_ref for i in (1.0, 0.1, 10.0, 0.01)[:])
sr_ref = SYSTEM_A_REFERENCE['sr']
sr_opts = tuple(i * sr_ref for i in (1, 0.25, 0.5)[:1])

SCRIPT = 'run_system_a.py'
OVERWRITE = False

DT_MAX = 0.01
COURANT_ADV = 0.5
COURANT_DIFF = 1.0
COURANT_REAC = 0.1
NX_SELECTOR = create_Nx_selector([160, 200], [1000.0])
NY_SELECTOR = NX_SELECTOR

N_STOP = 100_000
T_STOP = 120.0
WRITE_DELTA = (10, 5) if N_STOP > 100 else 1

NUMERICAL_PARAMS = FrozenDict(
    c_stabilization=None,
    c_limits=True,
)

DIR_ROOT = create_dir_path(
    NUMERICAL_PARAMS, 
    dir_root='./',
    dir_prefix='data', 
    dir_params=NUMERICAL_PARAMS.keys(), 
)
DIR_PARAMS = FrozenDict(
    dir_root=DIR_ROOT,
    dir_params=(*SYSTEM_A_REFERENCE.keys(), 'Nx', 'Ny'),
    dir_prefix=f't_stop={T_STOP}' if N_STOP > 100 else f'n_stop={N_STOP}',
    dir_suffix=None,
    dir_datetime=False,
    dir_uid=True,
)

CONFIG_PARAMS = FrozenDict(
    store_delta=None, 
    write_delta=WRITE_DELTA,
    **DIR_PARAMS,
)

SIMULATION_PARAMS = FrozenDict(
    **SYSTEM_A_REFERENCE,
    dt_max=DT_MAX,
    courant_adv=COURANT_ADV,
    courant_diff=COURANT_DIFF,
    courant_reac=COURANT_REAC,
    **NUMERICAL_PARAMS,
)
RUN_PARAMS = dict(
    n_stop=N_STOP,
    t_stop=T_STOP,
    dt_init=1e-6,
    n_init=20,
)
DEL_XDMF = False

opts_names = ('Ra', 'Da', 'sr')
opts_repr = lambda Ra, Da, sr:  ', '.join([f'{k}={v}' for k, v in zip(opts_names, (Ra, Da, sr))])
params_filtered: dict[tuple[float, float, float], dict] = {}

for Ra, Da, sr in combine_options(Ra_opts, Da_opts, sr_opts, link=False):
    simulation_params = SIMULATION_PARAMS.replace(
        Nx=NX_SELECTOR(Ra),
        Ny=NY_SELECTOR(Ra),
        Ra=Ra,
        Da=Da,
        sr=sr,
    )
    dir_path_partial = create_dir_path(
        simulation_params,
        **DIR_PARAMS.replace(dir_uid=False)
    )
    if not OVERWRITE and not dir_path_exists(dir_path_partial, glob_suffix='*'):
        print(f'Selected to run for n_stop={N_STOP}, t_stop={T_STOP}: {opts_repr(Ra, Da, sr)}')
        params_filtered[(Ra, Da, sr)] = simulation_params
    else:
        print(f'Already run for n_stop={N_STOP}, t_stop={T_STOP}: {opts_repr(Ra, Da, sr)}')

cli_params: list[list[tuple[str, str]]] = [] 
for (Ra, Da, sr), sim_params in params_filtered.items():
    cli_dict = dict(
        delete_xdmf=DEL_XDMF,
        **CONFIG_PARAMS,
        **sim_params,
        **RUN_PARAMS,
    )
    cli_params.append([(f'--{k}', repr(v)) for k, v in cli_dict.items()])

Selected to run for n_stop=100000, t_stop=120.0: Ra=100.0, Da=100.0, sr=0.2
Selected to run for n_stop=100000, t_stop=120.0: Ra=100.0, Da=10.0, sr=0.2
Selected to run for n_stop=100000, t_stop=120.0: Ra=100.0, Da=1000.0, sr=0.2
Selected to run for n_stop=100000, t_stop=120.0: Ra=100.0, Da=1.0, sr=0.2


## Run the simulations

In [4]:
N_PROC_MAX = 3
NICE = 0
LOG = True
LAZY_KWARGS = lambda: dict(
    start_new_session=True,
    close_fds=True,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.DEVNULL if not LOG else open("stdout.log", "w"),
    stderr=subprocess.DEVNULL if not LOG else open("stderr.log", "w"),
)

if N_PROC_MAX:
    commands = [
        f'python {SCRIPT} {" ".join(" ".join((str(name), repr(value))) for name, value in prm)}' for prm in cli_params
    ]
    p = subprocess.Popen(
        ['nohup', 'parallel', '--nice', repr(NICE), '-j', repr(N_PROC_MAX), ':::', *commands],
        **LAZY_KWARGS(),
    )
    print(f'Parent PID: {p.pid}')
else:
    for prm in cli_params:
        p = subprocess.Popen(
            [
                'nice', '-n', repr(NICE), 
                'nohup', 'python', SCRIPT,
                '--kill_pid', repr(True),
                *[i for pair in prm for i in pair],
            ],
            **LAZY_KWARGS(),
        )
        print(f'Child PID {p.pid}: {opts_repr(Ra, Da, sr)}')

Parent PID: 60720
